In [1]:
from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel, Field
from typing import List
import re

# TUTORING FLOW

input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph

In [ ]:

# --- 1. Helper Function untuk Membaca File ---
def read_text_file(file_path):
    """Membaca konten file teks untuk dijadikan knowledge base agent."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: File {file_path} tidak ditemukan. Pastikan file ada di folder yang sama."

# --- 1b. Chunking Functions ---
def chunk_pseudocode_kb(raw_text):
    """
    Memecah dokumen 'Pseudocode dan Golang Dasar.md' menjadi dict chunk tematik
    berdasarkan header '## <section>'.
    Return: dict[str, str] — key = nama section (lowercase), value = konten section.
    """
    sections = re.split(r'\n(?=## \d+-)', raw_text)
    chunks = {}
    for section in sections:
        # Ambil nama section dari header
        match = re.match(r'## \d+-(.+)', section.strip())
        if match:
            key = match.group(1).strip().lower().replace('-', '_')
            chunks[key] = section.strip()
        else:
            # Bagian awal (Daftar Isi, dst.) → simpan sebagai 'intro'
            if section.strip():
                chunks['intro'] = section.strip()
    return chunks

def chunk_misconceptions_kb(raw_text):
    """
    Memecah dokumen 'List of misconceptions.md' menjadi dict chunk tematik
    berdasarkan header '## <category>'.
    Return: dict[str, list[str]] — key = kategori, value = list item miskonsepsi.
    """
    categories = {}
    current_cat = None
    for line in raw_text.splitlines():
        header_match = re.match(r'^## (.+)', line.strip())
        if header_match:
            current_cat = header_match.group(1).strip()
            categories[current_cat] = []
        elif current_cat and line.strip().startswith('- '):
            categories[current_cat].append(line.strip())
    return categories

def strip_golang_examples(chunk_text):
    """
    Menghapus blok kode Go dan output dari chunk KB.
    Hanya menyisakan penjelasan teks + contoh pseudocode.
    """
    lines = chunk_text.split('\n')
    result = []
    skip = False
    for line in lines:
        # Skip blok ``` go ... ``` dan ``` plaintext ... ```
        if line.strip().startswith('``` go') or line.strip().startswith('```go'):
            skip = True
            continue
        if line.strip().startswith('``` plaintext') or line.strip().startswith('```plaintext'):
            skip = True
            continue
        if skip and line.strip() == '```':
            skip = False
            continue
        if skip:
            continue
        # Skip baris "**Output:**"
        if line.strip().startswith('**Output:**'):
            continue
        result.append(line)
    return '\n'.join(result)

def select_pseudocode_chunks(chunks, pseudocode_siswa, max_chars=4000):
    """
    Memilih chunk yang RELEVAN berdasarkan isi pseudocode siswa.
    Selalu sertakan: 'intro' (struktur program), 'variabel_dan_tipe_data', 'operator'.
    Tambahkan chunk lain berdasarkan keyword yang terdeteksi pada kode siswa.
    
    Args:
        max_chars: Batas maksimum karakter total yang di-inject (default 4000).
    """
    code = pseudocode_siswa.lower()

    # Chunk yang SELALU relevan untuk style checking
    always_relevant = ['intro', 'variabel_dan_tipe_data', 'operator']

    # Mapping keyword → chunk name
    keyword_map = {
        'output':       'output',
        'input':        'input_dan_output',
        'constant':     'konstanta',
        'for ':         'perulangan',
        'while ':       'perulangan',
        'repeat':       'perulangan',
        'endfor':       'perulangan',
        'endwhile':     'perulangan',
        'if ':          'percabangan',
        'else':         'percabangan',
        'endif':        'percabangan',
    }

    selected_keys = set(always_relevant)
    for kw, chunk_name in keyword_map.items():
        if kw in code:
            selected_keys.add(chunk_name)

    if 'perulangan' in selected_keys and 'percabangan' in selected_keys:
        selected_keys.add('perulangan_dan_percabangan')

    # Prioritas: always_relevant dulu, lalu sisanya
    ordered_keys = [k for k in always_relevant if k in chunks]
    ordered_keys += [k for k in selected_keys if k not in always_relevant and k in chunks]

    # Gabungkan chunk — strip Go examples — potong jika melebihi max_chars
    parts = []
    total_chars = 0
    for key in ordered_keys:
        cleaned = strip_golang_examples(chunks[key])
        if total_chars + len(cleaned) > max_chars:
            # Ambil sisa ruang yang tersedia
            remaining = max_chars - total_chars
            if remaining > 200:  # Minimal 200 char agar berguna
                parts.append(cleaned[:remaining] + "\n...[dipotong]")
            break
        parts.append(cleaned)
        total_chars += len(cleaned)

    return "\n\n---\n\n".join(parts)

def select_misconception_chunks(categories, pseudocode_siswa):
    """
    Memilih SUBSET miskonsepsi yang RELEVAN berdasarkan pola kode siswa.
    Tidak semua miskonsepsi di-inject — hanya yang cocok dengan construct yang ditemukan.
    
    Rules:
    - Ada 'while'/'endwhile' → sertakan miskonsepsi terkait loop kondisional
    - Ada 'if'/'endif'       → sertakan miskonsepsi terkait percabangan
    - Ada nested loop        → sertakan error nesting
    - Selalu sertakan        → Ketidaktelitian (carelessness) karena universal
    """
    code = pseudocode_siswa.lower()

    # Miskonsepsi terkait LOOP
    loop_misconceptions = {
        'While demon (WD)',
        'Intentional bug (IB)',
        'Mix of intentional bug and while demon (IBxWD)',
        'Conditional loop as conditional statement without alternative (WhileIf)',
        'Executed once (EO)',
    }

    # Miskonsepsi terkait PERCABANGAN / CONDITIONAL
    conditional_misconceptions = {
        'Conditional statement without alternative as conditional loop (IfWhile)',
        'Drop through error (DT)',
    }

    # Error terkait LOOP
    loop_errors = {
        'Full program as loop (SNIP)',
        'Execute n statement (EXN)',
        'Miscounting loop (Loop+ atau Loop-)',
    }

    # Error terkait NESTED LOOP
    nesting_errors = {
        'Ignore nesting 1 (IN1)',
        'Ignore nesting 2 (IN2)',
        'Ignore outer loop (IOL)',
        'Multiply counter (MC)',
    }

    # Deteksi pola pada kode siswa
    has_loop = any(kw in code for kw in ['while ', 'for ', 'repeat', 'endwhile', 'endfor'])
    has_conditional = any(kw in code for kw in ['if ', 'else', 'endif'])
    has_nested = code.count('for ') > 1 or code.count('while ') > 1 or \
                 ('for ' in code and 'while ' in code)

    # Bangun set nama miskonsepsi yang relevan
    relevant_names = set()
    if has_loop:
        relevant_names |= loop_misconceptions | loop_errors
    if has_conditional:
        relevant_names |= conditional_misconceptions
    if has_nested:
        relevant_names |= nesting_errors

    # Filter item dari setiap kategori
    result_parts = []
    for cat, items in categories.items():
        # Ketidaktelitian (Carelessness) → SELALU sertakan
        if 'ketidaktelitian' in cat.lower() or 'carelessness' in cat.lower():
            result_parts.append(f"## {cat}\n" + "\n".join(items))
            continue

        # Untuk kategori lain, filter berdasarkan nama miskonsepsi
        filtered = [item for item in items if any(name.lower() in item.lower() for name in relevant_names)]
        if filtered:
            result_parts.append(f"## {cat}\n" + "\n".join(filtered))

    if not result_parts:
        # Fallback: kembalikan semua jika tidak ada yang terdeteksi
        return "\n\n".join(f"## {cat}\n" + "\n".join(items) for cat, items in categories.items())

    return "\n\n".join(result_parts)


# --- Load & Chunk Knowledge Base ---
_raw_pseudocode = read_text_file("/home/ilham/Documents/python/crewai-vs-langgraph/doc/Pseudocode dan Golang Dasar.md")
_raw_misconceptions = read_text_file("/home/ilham/Documents/python/crewai-vs-langgraph/doc/List of misconceptions.md")

pseudocode_chunks = chunk_pseudocode_kb(_raw_pseudocode)
misconception_chunks = chunk_misconceptions_kb(_raw_misconceptions)

print(f"Pseudocode KB → {len(pseudocode_chunks)} chunk tematik: {list(pseudocode_chunks.keys())}")
print(f"Misconceptions KB → {len(misconception_chunks)} kategori: {list(misconception_chunks.keys())}")


# --- 2. Definisi Output Schema (Pydantic) ---
# Ini menjamin output JSON 100% valid sesuai format yang Anda minta
class AssessmentResult(BaseModel):
    score: int = Field(..., description="Nilai akhir siswa (integer 0-100)")
    correct: bool = Field(..., description="True jika logika benar, False jika salah")
    summary: str = Field(..., description="Ringkasan naratif penilaian")
    misconceptions: List[str] = Field(..., description="List nama miskonsepsi yang ditemukan (sesuai daftar referensi)")
    pseudocode: str = Field(..., description="Pseudocode ASLI siswa tanpa modifikasi")

# --- 3. Konfigurasi LLM ---
llm = LLM(
    model="ollama/llama3.1:8b",
    base_url="http://localhost:11434", # Updated: Removed /v1 for ollama provider
    temperature=0.1
)

# --- 4. Definisi Agents (Hierarchical) ---

# Sub-Agent 1: Style Checker
# Bertugas mengecek format berdasarkan dokumen 'Pseudocode dan Golang Dasar.md'
style_checker = Agent(
    role="Style & Syntax Auditor",
    goal="Memvalidasi kepatuhan pseudocode terhadap standar penulisan yang baku.",
    backstory=(
        "Anda adalah auditor kode yang sangat teliti terhadap sintaks.\n"
        "Anda memiliki akses ke dokumen 'Pseudocode dan Golang Dasar' sebagai acuan kebenaran.\n"
        "Tugas Anda:\n"
        "1. Memastikan struktur wajib: 'program', 'kamus', 'algoritma' ada.\n"
        "2. Memastikan deklarasi variabel menggunakan ':' dan tipe data yang benar (integer, real, string, boolean).\n"
        "3. Memastikan assignment menggunakan simbol '←' (panah kiri) bukan '='.\n"
        "4. Cek konsistensi penulisan variabel (case-sensitive) dan perintah I/O (input/output)."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=1
)

# Sub-Agent 2: Logic Checker
# Bertugas mengecek logika dan miskonsepsi berdasarkan 'List of misconceptions.md'
logic_checker = Agent(
    role="Logic & Misconception Analyst",
    goal="Menganalisis kebenaran algoritma dan mendeteksi pola miskonsepsi spesifik.",
    backstory=(
        "Anda adalah pakar logika algoritma dan pedagogi pemrograman.\n"
        "Anda memiliki referensi tentang miskonsepsi pemrograman yang RELEVAN dengan kode yang dianalisis.\n"
        "Tugas Anda:\n"
        "1. Bandingkan logika siswa dengan solusi referensi.\n"
        "2. Identifikasi jika siswa mengalami miskonsepsi spesifik dari daftar referensi yang diberikan.\n"
        "3. Tentukan apakah kode tersebut secara fungsional benar atau salah."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=1
)

# Manager: Scoring Supervisor
# Mengelola proses dan menghasilkan JSON final
scoring_supervisor = Agent(
    role="Scoring Supervisor",
    goal="Mengkonsolidasi laporan auditor, menghitung skor, dan menyusun JSON final.",
    backstory=(
        "Anda adalah Kepala Penilai.\n"
        "Tugas Anda adalah memimpin 'Style & Syntax Auditor' dan 'Logic & Misconception Analyst'.\n"
        "Anda tidak memeriksa kode sendiri, tetapi menggunakan laporan dari bawahan Anda untuk:\n"
        "1. Menghitung pengurangan skor berdasarkan rubrik.\n"
        "2. Menyusun kesimpulan (summary).\n"
        "3. Memastikan format output akhir adalah JSON yang valid."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=True, # Wajib True untuk Hierarchical Manager
    memory=True,
    max_iter=2
)

# --- 5. Definisi Tasks ---
# Seleksi chunk yang RELEVAN berdasarkan pseudocode siswa (dilakukan saat runtime via input_data)

# --- 7. Eksekusi ---
input_data = {
    'problem': "Buatlah algoritma untuk mencetak angka 1 sampai 5 menggunakan perulangan.",
    
    'context_solution': """
        program CetakAngka
        kamus
            i : integer
        algoritma
            for i <- 1 to 5 do
                output(i)
            endfor
        endprogram
    """,
    
    'pseudocode': """
        program CobaLoop
        kamus
            i : integer
        algoritma
            i = 1
            while i <= 5 do
                output(i)
                if i > 5 then
                    break
                endif
                i = i + 1
            endwhile
        endprogram
    """,
    
    'general_rubrication': """
        Start: 100
        - Salah Syntax (Assign pakai '=' bukan '<-'): -5 poin per kejadian
        - Logic Error (Loop logic aneh): -15 poin
        - Miskonsepsi terdeteksi: -10 poin
    """
}

# --- Seleksi Chunk RELEVAN berdasarkan pseudocode siswa ---
relevant_syntax_ref = select_pseudocode_chunks(pseudocode_chunks, input_data['pseudocode'])
relevant_misconceptions = select_misconception_chunks(misconception_chunks, input_data['pseudocode'])

print(f"Panjang referensi syntax yang di-inject: {len(relevant_syntax_ref)} chars "
      f"(dari {len(_raw_pseudocode)} chars total, hemat {100 - len(relevant_syntax_ref)*100//len(_raw_pseudocode)}%)")
print(f"Panjang referensi miskonsepsi yang di-inject: {len(relevant_misconceptions)} chars "
      f"(dari {len(_raw_misconceptions)} chars total, hemat {100 - len(relevant_misconceptions)*100//len(_raw_misconceptions)}%)")
print(f"\nMiskonsepsi yang di-inject:\n{relevant_misconceptions}")

# Task 1: Style Analysis — hanya inject chunk RELEVAN
task_style = Task(
    description=(
        "Analisis pseudocode siswa berikut berdasarkan referensi standar:\n\n"
        f"REFERENSI SYNTAX (chunk relevan saja):\n{relevant_syntax_ref}\n\n"
        "PSEUDOCODE SISWA:\n{pseudocode}\n\n"
        "Instruksi:\n"
        "1. Cek kelengkapan struktur (program, kamus, algoritma).\n"
        "2. Cek tipe data dan deklarasi variabel.\n"
        "3. Cek penggunaan operator assignment (←).\n"
        "Laporkan setiap pelanggaran sintaks."
    ),
    expected_output="Laporan detail pelanggaran style dan sintaks.",
    agent=style_checker,
    max_iter=1
)

# Task 2: Logic Analysis — hanya inject miskonsepsi yang RELEVAN
task_logic = Task(
    description=(
        "Analisis logika pseudocode siswa.\n\n"
        f"REFERENSI MISKONSEPSI (yang relevan dengan construct kode ini):\n{relevant_misconceptions}\n\n"
        "PROBLEM: {problem}\n"
        "SOLUSI REFERENSI: {context_solution}\n"
        "PSEUDOCODE SISWA: {pseudocode}\n\n"
        "Instruksi:\n"
        "1. Apakah logika siswa menghasilkan output yang sama dengan solusi?\n"
        "2. Apakah ditemukan pola miskonsepsi dari daftar referensi di atas? Fokus HANYA pada miskonsepsi yang relevan.\n"
        "3. Klasifikasikan kesalahan logika (Mayor/Minor)."
    ),
    expected_output="Laporan kebenaran logika dan daftar miskonsepsi yang terdeteksi.",
    agent=logic_checker,
    max_iter=1
)

# Task 3: Final Scoring & JSON Formatting
task_final = Task(
    description=(
        "Sebagai Supervisor, kumpulkan laporan dari Style dan Logic checker.\n"
        "Gunakan rubrik berikut untuk penilaian:\n{general_rubrication}\n\n"
        "Lakukan finalisasi:\n"
        "1. Hitung Score akhir (Start 100 dikurangi poin kesalahan).\n"
        "2. Tentukan status 'correct' (True/False).\n"
        "3. Ambil daftar 'misconceptions' yang valid dari laporan Logic Checker.\n"
        "4. Buat summary singkat.\n"
    ),
    expected_output="Objek JSON valid sesuai schema AssessmentResult.",
    agent=scoring_supervisor,
    context=[task_style, task_logic],
    output_pydantic=AssessmentResult, # Memaksa output JSON strict
    max_iter=2
)

# --- 6. Crew Assembly dengan Batasan Sistem ---

def step_callback(step):
    try:
        # Untuk event bertipe dict (lama / intermediate)
        if isinstance(step, dict):
            agent = step.get("agent_name", "UnknownAgent")
            task = step.get("task_description", "")
            count = step.get("step_count", "?")
            print(f"Iterasi {count}: {agent} - {task[:50]}...")
        else:
            # Untuk AgentAction / AgentFinish (object)
            print(f"{type(step).__name__} dari agent")
    except Exception as e:
        print(f"! step_callback error: {e}")

crew = Crew(
    agents=[style_checker, logic_checker],
    tasks=[task_style, task_logic, task_final],
    process=Process.hierarchical,
    manager_agent=scoring_supervisor,
    verbose=True,
    max_iter=3,  # Maksimal 3 iterasi untuk seluruh sistem
    step_callback=step_callback  # Monitor setiap langkah
)

# --- 7. Eksekusi dengan Error Handling ---

print("\n### MEMULAI PENILAIAN ###")
result = crew.kickoff(inputs=input_data)

print("\n### HASIL JSON FINAL ###")
# Karena menggunakan Pydantic, kita dump modelnya ke JSON string
print(result.pydantic.model_dump_json(indent=2))


Pseudocode KB → 10 chunk tematik: ['intro', 'output', 'variabel_dan_tipe_data', 'konstanta', 'komentar', 'input_dan_output', 'operator', 'perulangan', 'percabangan', 'perulangan_dan_percabangan']
Misconceptions KB → 3 kategori: ['Miskonsepsi (Misconception)', 'Eror (Error)', 'Ketidaktelitian (Carelessness)']
Panjang referensi syntax yang di-inject: 4028 chars (dari 28428 chars total, hemat 86%)
Panjang referensi miskonsepsi yang di-inject: 1801 chars (dari 2406 chars total, hemat 26%)

Miskonsepsi yang di-inject:
## Miskonsepsi (Misconception)
- Intentional bug (IB): Asumsi bahwa sistem dapat membuat pilihan berdasarkan keadaan masa depan mesin.
- While demon (WD): Eksekusi loop kondisional (perulangan bersyarat) dihentikan secara preventif berdasarkan perubahan kondisi keluar selama loop berjalan.
- Mix of intentional bug and while demon (IBxWD): Berhenti secara preventif di tengah jalan loop karena kondisi keluar akan berubah dengan eksekusi pernyataan berikutnya.
- Conditional loop 

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b56c7f93-6084-4021-8211-9cd03b8613ad                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scoring Supervisor                                                                                      │
│                                                                                                                 │
│  Task: Analisis pseudocode siswa berikut berdasarkan referensi standar:                                         │
│                                                                                                                 │
│  REFERENSI SYNTAX (chunk relevan saja):                                                                         │
│  # Pseudocode dan Golang Dasar                                                                                  │
│                                                                                                                 │
│  ## Daftar Isi                                                                                                  │
│                                                                                                                 │
│  1. [Output](#1-output)                                                                                         │
│  2. [Variabel dan Tipe Data](#2-variabel-dan-tipe-data)                                                         │
│  3. [Konstanta](#3-konstanta)                                                                                   │
│  4. [Komentar](#4-komentar)                                                                                     │
│  5. [Input dan Output](#5-input-dan-output)                                                                     │
│  6. [Operator](#6-operator)                                                                                     │
│  7. [Perulangan](#7-perulangan)                                                                                 │
│  8. [Percabangan](#8-percabangan)                                                                               │
│  9. [Perulangan dan Percabangan](#9-perulangan-dan-percabangan)                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2-variabel-dan-tipe-data                                                                                    │
│                                                                                                                 │
│  Saat mendeklarasikan variabel perlu dituliskan pula tipe data yang dapat ditampung oleh variabel itu. Tipe     │
│  data dasar (primitif) yang dikenal dalam bahasa pemrograman adalah numerik (bilangan bulat dan real), huruf    │
│  (string dan karakter) serta boolean. Cara mendeklarasikannya dapat dilihat pada Program 9-16, yaitu dengan     │
│  memberi tanda titik dua (:) di antara nama variabel dengan tipe datanya.                                       │
│                                                                                                                 │
│  Dalam pemrograman kita tidak boleh mendeklarasikan identifier, termasuk nama variabel, dengan menggunakan      │
│  keyword yang tersedia pada compiler. Berikut keyword dalam Golang yang tidak bisa dijadikan nama               │
│  (identifier): import, type, const, var, func, package, map, chan, struct, interface, if, else, for, range,     │
│  break, continue, goto, return, switch, case, select, default, fallthrough, defer, go.                          │
│                                                        

/home/ilham/Documents/python/crewai-vs-langgraph/.venv/lib/python3.11/site-packages/rich/live.py:256: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Style & Syntax Auditor                                                                                  │
│                                                                                                                 │
│  Task: Cek kelengkapan struktur (program, kamus, algoritma)                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

/home/ilham/Documents/python/crewai-vs-langgraph/.venv/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 3f640524-3f4c-4841-a9de-b6cf0830da5d                                                                     │
│  Agent: Scoring Supervisor                                                                                      │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: b56c7f93-6084-4021-8211-9cd03b8613ad                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output:                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ValidationError: 1 validation error for AssessmentResult
  Invalid JSON: key must be a string at line 1 column 2 [type=json_invalid, input_value="{'task': 'Kumpulkan lapo...tyle & Syntax Auditor'}", input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid

input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph